In [1]:
import pandas as pd
from carbon_forecast.evaluation.metrics import daily_regret

runs = {"all": "lgbm_l1_all_tuning", "from2024": "lgbm_l1_from2024_tuning"}
frames = []
for label, name in runs.items():
    p = pd.read_parquet(f"../data/predictions/{name}.parquet")
    valid = p[p["is_valid_target"]]
    month = valid["issue_time"].dt.strftime("%Y-%m")
    mae_m = (valid["y_pred"] - valid["y_true"]).abs().groupby(month).mean()
    r = daily_regret(p, 8)
    regret_m = r.groupby(r["issue_time"].dt.strftime("%Y-%m"))["regret"].mean()
    frames.append(pd.DataFrame({f"mae_{label}": mae_m, f"regret_{label}": regret_m}))

cmp = pd.concat(frames, axis=1).round(1)
cmp["mae_gap"] = (cmp["mae_from2024"] - cmp["mae_all"]).round(1)
cmp

,mae_all,regret_all,mae_from2024,regret_from2024,mae_gap
issue_time,,,,,
2024-10,18.2,6.7,19.3,4.7,1.1
2024-11,17.6,4.1,18.2,4.3,0.6
2024-12,16.6,6.7,23.5,6.4,6.9
2025-01,20.5,4.0,19.8,2.6,-0.7
2025-02,17.5,4.7,17.2,5.0,-0.3
2025-03,18.4,10.9,17.3,9.7,-1.1
2025-04,18.1,1.5,18.1,1.3,0.0
2025-05,17.3,3.6,16.9,2.4,-0.4
2025-06,17.1,4.8,18.8,4.8,1.7


**Takeaway:** The 2024+ model's higher tuning-year MAE (18.2 vs 17.3) is driven almost entirely by December 2024 (+6.9), the first December it had never seen in training. From January to September 2025 the two are tied (MAE 17.3 vs 17.4, regret 5.2 vs 5.1; monthly noise is 1 to 2 points). Decision: L1 on all history, which has seen every season including Christmas. A recency window must cover at least a full seasonal cycle.